In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import random
import numpy as np
import pickle

In [ ]:
#设置随机数种子用
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # 如果你使用的是多个GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super(TransformerEncoderLayer, self).__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        attn_output, _ = self.attention(x, x, x)
        x = x + self.dropout1(attn_output)
        x = self.norm1(x)
        ffn_output = self.ffn(x)
        x = x + self.dropout2(ffn_output)
        x = self.norm2(x)
        return x

class ChannelsAttenionlayer(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super(ChannelsAttenionlayer, self).__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x,channel_weight):
        attn_output, _ = self.attention(channel_weight, x, x)
        x = x + self.dropout1(attn_output)
        x = self.norm1(x)
        ffn_output = self.ffn(x)
        x = x + self.dropout2(ffn_output)
        x = self.norm2(x)
        return x
        

# 定义 Transformer 模型
class TransformerModel(nn.Module):
    def __init__(self, input_shape, embed_dim, num_heads, ff_dim, num_layers, num_outputs,dropout=0.1):
        super(TransformerModel, self).__init__()
        self.embedding = nn.Linear(input_shape[-1], embed_dim)
        self.channels_attentionlayer = ChannelsAttenionlayer(embed_dim, num_heads, ff_dim)
        self.dropout1 = nn.Dropout(dropout)
        
        self.translayers = nn.ModuleList(
            [TransformerEncoderLayer(embed_dim, num_heads, ff_dim) for _ in range(num_layers)]
        )
        
        self.fc = nn.Linear(embed_dim*input_shape[0], 640)
        self.fc2 = nn.Linear(640, 64)
        self.fc3 = nn.Linear(64, num_outputs)  

    def forward(self, x,channel_weights=None):
       
        x = self.embedding(x)  
        if channel_weights != None:
            channel_weights = channel_weights.unsqueeze(-1).expand_as(x)
            channel_weights = channel_weights.transpose(0,1)
            
        x = x.transpose(0, 1)   
        
        if channel_weights !=None:
            x = self.channels_attentionlayer(x,channel_weights)
        
        for layer in self.translayers:
            x = layer(x)
        
        x = x.transpose(0, 1)  

        x = torch.reshape(x,(len(x),-1))
        x = torch.relu(self.fc(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    

model = TransformerModel((64,55),25,5,64,1,2).to("cuda")

input_tensor = torch.randn(32, 64, 55).to("cuda")  

out = model(input_tensor)

out.shape

In [ ]:
with open('train_dataset.pkl', 'rb') as f:
    dataset = pickle.load(f)
f.close
with open('test_dataset.pkl', 'rb') as f:
    test_dataset = pickle.load(f)
f.close

set_seed(3047)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 

train_size = int(0.8 * len(dataset))  
val_size = len(dataset) - train_size  
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size],generator=torch.Generator().manual_seed(3047))


batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# 超参数
input_shape = (120, 56)
embed_dim = 35  # 嵌入维度
num_heads = 5  # 多头注意力的头数
ff_dim = 64  # 前馈网络的维度
num_layers = 2  # Transformer 编码器层的数量
num_outputs = 2  # 输出维度（x, y）

# 初始化模型
model = TransformerModel(input_shape, embed_dim, num_heads, ff_dim, num_layers, num_outputs)
criterion = CustomLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model.to(device)
epochs = 350

# 模型训练
min_loss = float('inf')
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for i, data in enumerate(train_loader):
        # print(data.shape)
        
        labels, corr_datas,inputs = data[:,:2],data[:,2:122].unsqueeze(2),data[:,122:].view(len(data),120,55)
        # print(corr_datas.shape,inputs.shape)
        inputs = torch.cat((corr_datas,inputs),dim=-1)
        # print(inputs.shape)
        
        inputs, labels= inputs.to(device),labels.to(device)  # 将数据移动到 device 上
        
        optimizer.zero_grad()
        # print(inputs.shape)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        # if i % 100 == 99:    # 每100个小批量打印一次
        #     print(f'[Epoch {epoch + 1}, Batch {i + 1}] loss: {running_loss / 100:.3f}')
        #     running_loss = 0.0
    
    # 每个epoch 都验证一次结果，结果最优即保存model
    val_loss = test(model,criterion,val_loader,device)
    if val_loss < min_loss:
        min_loss = val_loss
        torch.save(model.state_dict(), 'Trans_seed3047.pth')
    print(f'[Epoch {epoch + 1}]  train_loss:{running_loss/train_loader.__len__():.3f}; val_loss: {val_loss:.3f}')
            

print('Finished Training')

In [ ]:

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 创建一个新的模型实例
# 超参数
input_shape = (64, 56)
embed_dim = 25  # 嵌入维度
num_heads = 5  # 多头注意力的头数
ff_dim = 64  # 前馈网络的维度
num_layers = 2  # Transformer 编码器层的数量
num_outputs = 2  # 输出维度（x, y）

# 初始化模型
model_loaded = TransformerModel(input_shape, embed_dim, num_heads, ff_dim, num_layers, num_outputs)

# 加载模型参数 
model_loaded.load_state_dict(torch.load('Trans_seed3047.pth'))
model_loaded.to(device)
model_loaded.eval()

# 运行训练和测试
test(model_loaded,criterion,test_loader,device=device)